# Session Index and Trial Catalog for Monkey N

This notebook converts the NWB-level exploratory outputs into a clean, decoder-ready metadata layer.

## Objectives
1. Load the outputs generated by Notebook 01 (EDA).
2. Build a canonical session index.
3. Extract and standardize trial tables from all NWB sessions.
4. Summarize target-style composition, trial timing, and session-level consistency.
5. Export tables for later decoding and adaptive-threshold notebooks.

## Inputs
- `/kaggle/working/tables_eda/session_inventory.csv`
- `/kaggle/working/tables_eda/nwb_session_summary.csv`
- `/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N/*.nwb`

## Outputs
- Session master table
- Trial catalog across all sessions
- Session-level trial summary
- QC figures for longitudinal and task-structure analysis

In [ ]:
!pip install pynwb h5py

In [ ]:
import os
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from pynwb import NWBHDF5IO

In [ ]:
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "axes.linewidth": 1.2,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "grid.linestyle": ":",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.85,
})

def paper_axes(ax):
    ax.minorticks_on()
    ax.grid(True, which="major", linestyle=":", linewidth=0.8)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.7)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
    ax.tick_params(which="both", direction="in", top=True, right=True)

## Paths

This notebook reuses the exact Kaggle dataset path and the generated EDA outputs from Notebook 01.

In [ ]:
DATASET_DIR = Path("/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N")

EDA_TABLE_DIR = Path("/kaggle/working/tables_eda")
EDA_META_DIR = Path("/kaggle/working/meta_eda")

WORK_ROOT = Path("/kaggle/working")
OUT_TABLE_DIR = WORK_ROOT / "tables_session_index"
OUT_FIG_DIR = WORK_ROOT / "figures_session_index"
OUT_META_DIR = WORK_ROOT / "meta_session_index"

OUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_META_DIR.mkdir(parents=True, exist_ok=True)

SESSION_INVENTORY_CSV = EDA_TABLE_DIR / "session_inventory.csv"
SESSION_SUMMARY_CSV = EDA_TABLE_DIR / "nwb_session_summary.csv"

print("DATASET_DIR:", DATASET_DIR)
print("SESSION_INVENTORY_CSV:", SESSION_INVENTORY_CSV)
print("SESSION_SUMMARY_CSV:", SESSION_SUMMARY_CSV)

In [ ]:
assert DATASET_DIR.exists(), f"Missing dataset directory: {DATASET_DIR}"
assert SESSION_INVENTORY_CSV.exists(), f"Missing EDA file: {SESSION_INVENTORY_CSV}"
assert SESSION_SUMMARY_CSV.exists(), f"Missing EDA file: {SESSION_SUMMARY_CSV}"

inventory_df = pd.read_csv(SESSION_INVENTORY_CSV)
session_df = pd.read_csv(SESSION_SUMMARY_CSV)

inventory_df["session_date"] = pd.to_datetime(inventory_df["session_date"], errors="coerce")
session_df["session_date"] = pd.to_datetime(session_df["session_date"], errors="coerce")

print("inventory_df:", inventory_df.shape)
print("session_df:", session_df.shape)

display(inventory_df.head())
display(session_df.head())

In [ ]:
def extract_session_date_from_name(file_name):
    m = re.search(r"_ses-(\d{8})_", str(file_name))
    if m is None:
        return pd.NaT
    return pd.to_datetime(m.group(1), format="%Y%m%d", errors="coerce")

def infer_target_style(row):
    vals = []
    for col in ["session_description", "identifier", "file_name"]:
        val = row.get(col, None)
        if pd.notna(val):
            vals.append(str(val).lower())

    joined = " ".join(vals)

    if " target style co" in joined or "_co_" in joined or joined.endswith("_co_nwb"):
        return "CO"
    if " target style rd" in joined or "_rd_" in joined or joined.endswith("_rd_nwb"):
        return "RD"

    m = re.search(r"target style\s+([a-z0-9]+)", joined)
    if m:
        return m.group(1).upper()

    return "UNKNOWN"

def safe_read_trials(nwb_path):
    with NWBHDF5IO(str(nwb_path), mode="r", load_namespaces=True) as io:
        nwb = io.read()
        if nwb.trials is None:
            return pd.DataFrame()
        return nwb.trials.to_dataframe().reset_index()

## Build the session master table

This table becomes the canonical reference for every later notebook.

In [ ]:
session_master_df = session_df.copy()

session_master_df["file_name"] = session_master_df["file_name"].astype(str)
session_master_df["session_date"] = session_master_df["session_date"].fillna(
    session_master_df["file_name"].apply(extract_session_date_from_name)
)

session_master_df["target_style_inferred"] = session_master_df.apply(infer_target_style, axis=1)
session_master_df["has_trials"] = session_master_df["n_trials"].fillna(0).astype(int) > 0
session_master_df["has_electrodes"] = session_master_df["n_electrodes"].fillna(0).astype(int) > 0
session_master_df["has_units"] = session_master_df["n_units"].fillna(0).astype(int) > 0

keep_cols = [
    "file_name", "file_path", "session_date", "identifier", "session_description",
    "session_start_time", "institution", "lab", "subject_id", "subject_species",
    "n_trials", "n_units", "n_electrodes", "n_acquisition_streams",
    "n_processing_modules", "n_intervals_tables", "trials_present",
    "units_present", "electrodes_present", "target_style_inferred"
]

session_master_df = session_master_df[keep_cols].sort_values("session_date").reset_index(drop=True)
session_master_df["session_index"] = np.arange(1, len(session_master_df) + 1)

if session_master_df["session_date"].notna().any():
    first_date = session_master_df["session_date"].min()
    session_master_df["days_since_first_session"] = (
        session_master_df["session_date"] - first_date
    ).dt.days
else:
    session_master_df["days_since_first_session"] = np.nan

session_master_df.to_csv(OUT_TABLE_DIR / "session_master_index.csv", index=False)
display(session_master_df.head(15))

In [ ]:
style_counts = session_master_df["target_style_inferred"].value_counts(dropna=False).rename_axis("target_style").reset_index(name="n_sessions")
style_counts.to_csv(OUT_TABLE_DIR / "target_style_session_counts.csv", index=False)
display(style_counts)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

for style, subdf in session_master_df.groupby("target_style_inferred"):
    ax.scatter(
        subdf["session_date"],
        subdf["session_index"],
        s=34,
        label=style,
        alpha=0.9
    )

ax.set_xlabel("Session date")
ax.set_ylabel("Session index")
ax.set_title("Longitudinal session coverage by inferred target style", pad=10)
paper_axes(ax)
ax.legend(loc="best")

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig01_session_coverage_by_target_style.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig01_session_coverage_by_target_style.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Extract the full trial catalog

We now read the `trials` table from every NWB file and standardize the columns into one combined trial-level table.

In [ ]:
nwb_files = sorted(DATASET_DIR.glob("*_ecephys.nwb"))
assert len(nwb_files) > 0, "No NWB files found."

session_lookup = session_master_df.set_index("file_name").to_dict(orient="index")

trial_tables = []

for i, nwb_path in enumerate(nwb_files, start=1):
    if i % 25 == 0 or i == 1 or i == len(nwb_files):
        print(f"Reading trials {i}/{len(nwb_files)}: {nwb_path.name}")

    trials_df = safe_read_trials(nwb_path)

    if trials_df.empty:
        continue

    meta = session_lookup.get(nwb_path.name, {})
    trials_df["file_name"] = nwb_path.name
    trials_df["file_path"] = str(nwb_path)
    trials_df["session_date"] = meta.get("session_date", pd.NaT)
    trials_df["session_index"] = meta.get("session_index", np.nan)
    trials_df["identifier"] = meta.get("identifier", None)
    trials_df["target_style_inferred"] = meta.get("target_style_inferred", "UNKNOWN")

    if "start_time" in trials_df.columns and "stop_time" in trials_df.columns:
        trials_df["trial_duration"] = trials_df["stop_time"] - trials_df["start_time"]
    else:
        trials_df["trial_duration"] = np.nan

    if "timeseries" in trials_df.columns:
        trials_df["timeseries_n_items"] = trials_df["timeseries"].apply(
            lambda x: len(x) if hasattr(x, "__len__") else np.nan
        )

    trial_tables.append(trials_df)

trial_catalog_df = pd.concat(trial_tables, ignore_index=True, sort=False)
trial_catalog_df["session_date"] = pd.to_datetime(trial_catalog_df["session_date"], errors="coerce")

print("trial_catalog_df shape:", trial_catalog_df.shape)
display(trial_catalog_df.head())

In [ ]:
preferred_col_order = [
    "file_name", "session_index", "session_date", "identifier", "target_style_inferred",
    "id", "start_time", "stop_time", "trial_duration",
    "trial_number", "trial_count", "run_id",
    "index_target_position", "mrs_target_position", "target_style",
    "trial_timeout", "timeseries_n_items", "file_path"
]

existing_cols = [c for c in preferred_col_order if c in trial_catalog_df.columns]
remaining_cols = [c for c in trial_catalog_df.columns if c not in existing_cols]
trial_catalog_df = trial_catalog_df[existing_cols + remaining_cols]

trial_catalog_df.to_csv(OUT_TABLE_DIR / "trial_catalog_all_sessions.csv", index=False)
display(trial_catalog_df.head(10))

## Session-level trial summary

This condenses the trial catalog into one row per session for later filtering and train/test bookkeeping.

In [ ]:
agg_dict = {
    "id": "count",
    "trial_duration": ["mean", "median", "std", "min", "max"],
}

for optional_col in ["trial_number", "trial_count", "run_id", "index_target_position", "mrs_target_position", "target_style", "timeseries_n_items"]:
    if optional_col in trial_catalog_df.columns:
        agg_dict[optional_col] = "nunique"

session_trial_summary = trial_catalog_df.groupby(
    ["file_name", "session_index", "session_date", "target_style_inferred"],
    dropna=False
).agg(agg_dict)

session_trial_summary.columns = [
    "_".join([str(x) for x in col if str(x) != ""]).strip("_")
    for col in session_trial_summary.columns.to_flat_index()
]

session_trial_summary = session_trial_summary.reset_index().rename(columns={"id_count": "n_trials_from_catalog"})
session_trial_summary = session_trial_summary.sort_values("session_date").reset_index(drop=True)

session_trial_summary.to_csv(OUT_TABLE_DIR / "session_trial_summary.csv", index=False)
display(session_trial_summary.head(15))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 10), sharex=True)

axes[0].plot(
    session_trial_summary["session_index"],
    session_trial_summary["n_trials_from_catalog"],
    color="#4C72B0",
    lw=2,
    marker="o",
    ms=4,
    markerfacecolor="white"
)
axes[0].set_ylabel("Trials per session")
axes[0].set_title("Session-wise trial counts", pad=8)
paper_axes(axes[0])

axes[1].plot(
    session_trial_summary["session_index"],
    session_trial_summary["trial_duration_mean"],
    color="#8172B2",
    lw=2,
    marker="s",
    ms=4,
    markerfacecolor="white"
)
axes[1].set_xlabel("Session index")
axes[1].set_ylabel("Mean trial duration [s]")
axes[1].set_title("Mean trial duration by session", pad=8)
paper_axes(axes[1])

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig02_trial_count_and_duration_by_session.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig02_trial_count_and_duration_by_session.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

ax.hist(
    trial_catalog_df["trial_duration"].dropna(),
    bins=50,
    color="#937860",
    edgecolor="black",
    linewidth=0.6,
    alpha=0.9
)

ax.set_xlabel("Trial duration [s]")
ax.set_ylabel("Count")
ax.set_title("Distribution of trial durations across all sessions", pad=10)
paper_axes(ax)

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig03_all_trial_durations.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig03_all_trial_durations.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Column stability and missingness

This step checks whether the trial schema is stable enough for a single downstream preprocessing pipeline.

In [ ]:
trial_column_presence = pd.DataFrame({
    "column": trial_catalog_df.columns,
    "n_missing": trial_catalog_df.isna().sum().values,
    "pct_missing": (100 * trial_catalog_df.isna().mean()).round(2).values,
    "n_unique": [trial_catalog_df[c].nunique(dropna=True) for c in trial_catalog_df.columns],
    "dtype": trial_catalog_df.dtypes.astype(str).values,
}).sort_values(["pct_missing", "n_unique"], ascending=[False, False])

trial_column_presence.to_csv(OUT_TABLE_DIR / "trial_column_presence.csv", index=False)
display(trial_column_presence.head(25))

In [ ]:
core_cols = [c for c in [
    "start_time", "stop_time", "trial_duration", "trial_number", "trial_count",
    "run_id", "index_target_position", "mrs_target_position", "target_style",
    "trial_timeout", "timeseries_n_items"
] if c in trial_catalog_df.columns]

presence_matrix = trial_catalog_df[core_cols].notna().astype(int)

fig, ax = plt.subplots(figsize=(13, max(4, 0.45 * len(core_cols))))
im = ax.imshow(presence_matrix.T.values, aspect="auto", cmap="Greys", vmin=0, vmax=1)

ax.set_yticks(range(len(core_cols)))
ax.set_yticklabels(core_cols)
ax.set_xlabel("Trial row index")
ax.set_title("Presence/absence matrix for core trial columns", pad=10)
paper_axes(ax)

cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Present (1) / Missing (0)")

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig04_trial_column_presence_matrix.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig04_trial_column_presence_matrix.pdf", dpi=300, bbox_inches="tight")
plt.show()